# `py-<PkgName>` — R parity validation

Side-by-side comparison of `py<pkgname>` against R `<UpstreamR>` on the canonical fixture declared in `data/manifest.yaml`.

This notebook follows the 6-section schema from [omicverse-rebuildr/NOTEBOOKS.md](https://github.com/omicverse/omicverse-rebuildr/blob/main/NOTEBOOKS.md): setup → R reference → Python candidate → per-output parity → wall-clock → verdict.

Pre-executed and committed for GitHub preview. Re-run before each release with:

```bash
jupyter nbconvert --to notebook --execute examples/compare_R_vs_Python.ipynb --output compare_R_vs_Python.ipynb
```

## 1. Setup

In [ ]:
import os, subprocess, time, json
# Lock BLAS thread count BEFORE numpy import
for k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[k] = '8'

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PORT_DIR = Path('.').resolve().parent  # ran from examples/
manifest = yaml.safe_load((PORT_DIR / 'data' / 'manifest.yaml').read_text())
print(f"Port:        {manifest['package']}")
print(f"Upstream:    {manifest['upstream']['name']} v{manifest['upstream']['version']}")
print(f"Class:       {manifest['algorithm_class']}")
print(f"Threshold:   {manifest['parity_threshold']}")
print(f"Fixture:     {manifest['fixture']['path']}")
print(f"Seed:        {manifest['seed']}")

## 2. R reference run

In [ ]:
R_ENV = os.environ.get('R_TEST_ENV', '/path/to/your/R/env')
ref_out = PORT_DIR / 'data' / 'reference_output.json'
fixture = PORT_DIR / manifest['fixture']['path']

t0 = time.perf_counter()
subprocess.run(
    ['conda', 'run', '-p', R_ENV, 'Rscript', str(PORT_DIR / manifest['reference_command']),
     str(fixture), str(ref_out)],
    check=True, cwd=PORT_DIR,
)
r_wall = time.perf_counter() - t0
ref = json.loads(ref_out.read_text())
print(f"R wall-clock: {r_wall:.2f}s")

## 3. Python candidate run

In [ ]:
cand_out = PORT_DIR / 'data' / 'candidate_output.json'
t0 = time.perf_counter()
subprocess.run(
    ['python', str(PORT_DIR / 'tests' / '_run_candidate.py'),
     str(fixture), str(cand_out)],
    check=True, cwd=PORT_DIR,
)
py_wall = time.perf_counter() - t0
cand = json.loads(cand_out.read_text())
print(f"Py wall-clock: {py_wall:.2f}s   (speedup vs R: {r_wall/py_wall:.1f}×)")

## 4. Per-output parity

One subsection per `manifest.yaml::outputs[]` block. Replace the placeholder cells below with the visualisation appropriate to each output's `algorithm_class` (see `NOTEBOOKS.md §Notebook 1 step 4`).

Patterns by class:
- `deterministic`: line plot R vs Py overlay + max abs err
- `clustering`: confusion-matrix heatmap + ARI
- `embedding`: side-by-side scatter + Procrustes
- `ordinal`: scatter (R vs Py) + Pearson + Spearman
- `classification`: confusion-matrix heatmap + F1
- `ranked`: top-K Jaccard + Venn
- `inference`: -log10(p) scatter + top-K overlap
- `stochastic`: KDE overlay + KS p-value

In [ ]:
# TODO — fill in one visualisation per output block in manifest.outputs.
# Use omicverse-rebuildr/engine/parity_metrics.py for the metric computation.
import sys
sys.path.insert(0, 'omicverse-rebuildr/engine')
from parity_metrics import compute_parity, is_pass

results = []
for spec in manifest.get('outputs', []):
    name = spec['name']
    ref_key = spec['location_reference'].lstrip('$.')
    cand_key = (spec['location_candidate'].split('[')[-1].strip("]'\"")
                if '[' in spec['location_candidate']
                else spec['location_candidate'].rsplit('.', 1)[-1])
    cls = spec.get('metric', manifest['algorithm_class'])
    threshold = spec.get('threshold', manifest['parity_threshold'])
    metric = compute_parity(ref[ref_key], cand[cand_key], cls)
    passed = is_pass(metric, cls, threshold)
    results.append((name, cls, threshold, metric, passed))
    print(f"{name:20s} class={cls:12s} threshold={threshold}  measured={metric}  {'PASS' if passed else 'FAIL'}")


## 5. Wall-clock comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(['R reference', f'py{manifest["package"].lower().replace("py-", "")}'],
        [r_wall, py_wall], color=['#a4262c', '#0078d4'])
ax.set_xlabel('Wall-clock (s)')
ax.set_title(f'Speedup: {r_wall/py_wall:.1f}× on {manifest["fixture"]["path"]}')
for i, t in enumerate([r_wall, py_wall]):
    ax.text(t, i, f' {t:.2f}s', va='center')
plt.tight_layout(); plt.show()

## 6. Verdict

In [ ]:
all_pass = all(r[4] for r in results)
print('━' * 60)
for name, cls, thr, m, p in results:
    print(f'  {"✅" if p else "❌"}  {name:20s}  class={cls:10s}  thr={thr}  measured={m}')
print('━' * 60)
print(('PASS' if all_pass else 'FAIL') +
      f' — pre-registered gate {"cleared" if all_pass else "NOT cleared"} on {manifest["fixture"]["path"]}.')